# Chapter 3 Practical 06: Cold Start, Sparsity, Clustering, and Temporal Dynamics

Learning objectives:
- Detect cold-start users and items.
- Discuss sparsity, popularity bias, and cold start.
- Cluster users to reduce neighbor search.
- Add time-decay weights to recent interactions.
- Complete three short Chapter 3 challenges.

Slide connection: limits of memory-based CF, practical mitigations, clustering for scalability, and temporal dynamics.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_03_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_03_collaborative_filtering/data"

def read_chapter3_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter3_csv("ratings_chapter3.csv")
movies = read_chapter3_csv("movies_chapter3.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter3.csv from data/ratings_chapter3.csv
Loaded movies_chapter3.csv from data/movies_chapter3.csv


title,Blade Runner,Finding Nemo,Independence Day,Jurassic Park,Star Wars,Terminator 2,The Matrix,The Notebook,Titanic,Toy Story
user_id,,,,,,,,,,
Alice,5.0,NaN,4.0,NaN,4.0,NaN,5.0,NaN,NaN,NaN
Bob,NaN,NaN,6.0,4.0,7.0,4.0,7.0,NaN,NaN,NaN
Chris,NaN,NaN,2.0,7.0,3.0,7.0,NaN,NaN,NaN,5.0
Karen,NaN,NaN,NaN,4.0,7.0,3.0,6.0,NaN,NaN,NaN
Lynn,NaN,NaN,2.0,4.0,4.0,6.0,NaN,NaN,NaN,6.0
Nina,NaN,5.0,NaN,4.0,NaN,NaN,NaN,NaN,2.0,5.0
Omar,NaN,2.0,NaN,3.0,NaN,NaN,NaN,5.0,5.0,NaN
Sally,NaN,NaN,7.0,6.0,7.0,3.0,6.0,NaN,NaN,NaN


## Packages and key functions used

- `pandas` is used for tabular data. Important functions in this notebook include `read_csv`, `merge`, `pivot_table`, `groupby`, `agg`, `sort_values`, and `dropna`.
- `numpy` is used for numerical operations such as vector norms, dot products, averages, absolute values, and exponential time-decay weights.
- `pathlib.Path` helps the notebook find local CSV files in Colab, Jupyter, or the repository folder.
- The shared helper `read_chapter3_csv()` first looks for local data files and then falls back to the GitHub raw URL when students open the notebook directly online.
- `rating_matrix` is the central memory-based collaborative filtering object: rows are users, columns are movies, observed numbers are ratings, and missing cells mean unknown preferences.


Cold-start users and cold-start items have too few interactions for reliable memory-based CF.


In [2]:
user_counts = ratings_named.groupby("user_id").size().rename("ratings_count")
item_counts = ratings_named.groupby("title").size().rename("ratings_count")

print("Cold-start-like users:")
display(user_counts[user_counts <= 2])
print("Cold-start-like items:")
display(item_counts[item_counts <= 2])


Cold-start-like users:


Series([], Name: ratings_count, dtype: int64)

Cold-start-like items:


title
Blade Runner    1
Finding Nemo    2
The Notebook    1
Titanic         2
Name: ratings_count, dtype: int64

## Technique: sparsity and popularity bias

Sparsity means that most user-item pairs are unknown. Popularity bias means popular items get more evidence and may be recommended more often.

Important functions:

- `groupby("title")["rating"].agg(...)` calculates item-level counts and means.
- `notna().sum().sum()` counts observed ratings.
- Sorting by `ratings_count` helps identify items that dominate the available evidence.


In [3]:
n_users, n_items = rating_matrix.shape
known_ratings = rating_matrix.notna().sum().sum()
sparsity = 1 - known_ratings / (n_users * n_items)

popularity = (
    ratings_named.groupby("title")["rating"]
    .agg(ratings_count="count", mean_rating="mean")
    .sort_values(["ratings_count", "mean_rating"], ascending=False)
)

print(f"Matrix sparsity: {sparsity:.1%}")
popularity.head(5).round(2)


Matrix sparsity: 55.0%


,ratings_count,mean_rating
title,,
Jurassic Park,7,4.57
Star Wars,6,5.33
Terminator 2,5,4.60
Independence Day,5,4.20
The Matrix,4,6.00


## Technique: clustering to reduce neighbor search

One scalability mitigation is to compare the target user only with users in the same cluster.

This section uses `scikit-learn`:

- `StandardScaler` puts filled rating columns on a comparable scale.
- `KMeans` groups users with similar filled rating profiles.
- Missing values are filled with item means only for clustering. They are not treated as real ratings for CF prediction.


In [4]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

filled = rating_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)
scaled = StandardScaler().fit_transform(filled)
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
clusters = pd.Series(kmeans.fit_predict(scaled), index=rating_matrix.index, name="cluster")
clusters.to_frame().sort_values("cluster")


,cluster
user_id,
Omar,0
Alice,1
Chris,1
Lynn,1
Bob,2
Karen,2
Nina,2
Sally,2


In [5]:
target_user = "Karen"
target_cluster = clusters.loc[target_user]
candidate_neighbors = clusters[clusters.eq(target_cluster)].index.drop(target_user)
print(f"Compare Karen only with users in cluster {target_cluster}: {candidate_neighbors.tolist()}")


Compare Karen only with users in cluster 2: ['Bob', 'Nina', 'Sally']


## Technique: temporal dynamics and time decay

Temporal dynamics give more weight to recent interactions. A larger decay value makes older ratings fade more quickly.

Important functions:

- `np.exp(-decay_lambda * days_ago)` creates a smooth time-decay weight.
- `np.average(..., weights=...)` calculates a weighted mean rating.
- The `recent_profile` table shows which movies look stronger when recent ratings count more.


In [6]:
decay_lambda = 0.01
ratings_named["time_weight"] = np.exp(-decay_lambda * ratings_named["days_ago"])
ratings_named["weighted_rating"] = ratings_named["rating"] * ratings_named["time_weight"]

ratings_named[["user_id", "title", "rating", "days_ago", "time_weight", "weighted_rating"]].sort_values("days_ago").head(10).round(3)


,user_id,title,rating,days_ago,time_weight,weighted_rating
33,Omar,The Notebook,5,6,0.942,4.709
9,Bob,The Matrix,7,6,0.942,6.592
29,Nina,Finding Nemo,5,7,0.932,4.662
23,Karen,The Matrix,6,8,0.923,5.539
19,Lynn,Toy Story,6,10,0.905,5.429
18,Lynn,Independence Day,2,12,0.887,1.774
25,Alice,Blade Runner,5,12,0.887,4.435
4,Sally,The Matrix,6,14,0.869,5.216
32,Omar,Titanic,5,15,0.861,4.304
3,Sally,Independence Day,7,20,0.819,5.731


In [7]:
recent_profile = (
    ratings_named.groupby("title")
    .apply(lambda g: np.average(g["rating"], weights=g["time_weight"]))
    .rename("time_weighted_mean")
    .sort_values(ascending=False)
)
recent_profile.head(5).round(2)


/var/folders/w0/2jgnn0bx27b_jyy4mrm45cxm0000gn/T/ipykernel_69128/2327955874.py:2: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  ratings_named.groupby("title")


title
The Matrix      6.08
Toy Story       5.37
Blade Runner    5.00
The Notebook    5.00
Star Wars       4.77
Name: time_weighted_mean, dtype: float64

# Challenges

### Challenge 1 — Change the Number of Clusters

**Goal:**
Investigate how clustering changes the reduced neighbor search space.

**What to do:**

1. Change `n_clusters=3` in the `KMeans` cell to `2`.
2. Rerun the clustering and candidate-neighbor cells.
3. Change `n_clusters` to `4` and rerun the same cells.
4. Compare the candidate neighbors for `target_user`.


In [8]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

How did the candidate-neighbor list change? Which cluster setting gave a useful search space? Why can too many or too few clusters be a problem?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 — Change the Time-Decay Rate

**Goal:**
Investigate how stronger or weaker time decay changes the time-weighted popularity ranking.

**What to do:**

1. Change `decay_lambda` from `0.01` to `0.03`.
2. Rerun the time-weight and `recent_profile` cells.
3. Change `decay_lambda` to `0.001` and rerun the same cells.
4. Compare which movies move up or down in the time-weighted ranking.


In [9]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which movies benefited most from stronger time decay? Which older interactions lost influence? When would time decay be useful in a recommender?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 — Concept Check: Cold Start and Popularity Bias

This challenge requires **no programming**.

> A new user joins a movie platform but has not rated, liked, or watched any movies.
> At the same time, a newly released movie has not yet received any interactions.

Explain in your own words:

1. Why can standard memory-based collaborative filtering not provide reliable personalized recommendations in these two cases?
2. Suggest one practical solution for the new user.
3. Suggest one practical solution for the new item.
4. Why can always falling back to popular items reduce recommendation diversity?

### Your explanation
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
